# Gym Clean food classifier: train on Colab (public data only)

Trains the M7b Thai food classifier on **THFOOD-50** with a Colab GPU and gives you a `.tflite`
to import in the app (Settings → Food photos → Import model).

- **Public data only.** Your own meal photos are never uploaded here: they train on your Mac only
  (see `ml/README.md`, "Mac").
- **THFOOD-50 licence:** non-commercial research/educational use only; the images belong to third
  parties. The dataset is downloaded into this Colab machine (or your Google Drive), never into git,
  and the trained model is for your own use: don't publish it.
- Set `ACCEPT_THFOOD50_LICENCE = True` below only after reading the terms. Then *Runtime → Run all*.
- **Disconnects:** with `USE_DRIVE = True`, data, embeddings and the head checkpoint live in Drive,
  so *Run all* again resumes where it stopped.
- Use a GPU runtime (*Runtime → Change runtime type → T4 GPU*). About 15–30 min end to end.

In [ ]:
# --- Settings -------------------------------------------------------------------------------
ACCEPT_THFOOD50_LICENCE = False   # set True only after reading https://github.com/chakkritte/THFOOD-50
USE_DRIVE = True                  # keep data + checkpoints in Google Drive so a disconnect can resume
RUN_ID = "colab-1"                # same id = resume; new id = fresh head (embeddings are reused)
EPOCHS = 30                       # max head epochs (early stopping on val may end sooner)
REPO = "https://github.com/wongsatonk/gym-clean"
BRANCH = "m7b-food-vision"

In [ ]:
import os, subprocess, sys

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/gym-clean-ml"
else:
    WORK = "/content/gym-clean-ml"
os.makedirs(WORK, exist_ok=True)
CODE = "/content/gym-clean"   # the repo checkout (code only; nothing here is committed back)

def run(*args):
    """Runs a command, streaming its output; on failure shows the last lines (the error) and stops."""
    args = [str(a) for a in args]
    print("$", " ".join(args), flush=True)
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in p.stdout:
        print(line, end="", flush=True)
        lines.append(line)
    if p.wait() != 0:
        tail = "".join(lines[-40:])
        raise RuntimeError(f"command failed (exit {p.returncode}): {' '.join(args)}\n--- last output ---\n{tail}")

if not os.path.isdir(CODE):
    run("git", "clone", "--depth", "1", "--branch", BRANCH, REPO, CODE)
else:
    run("git", "-C", CODE, "pull", "--ff-only")
print("work dir:", WORK)

In [ ]:
# Pinned extras (Colab keeps its own GPU TensorFlow).
run(sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE}/ml/requirements-colab.txt")
import tensorflow as tf
print("TensorFlow", tf.__version__, "GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
# Licence: read it, then set ACCEPT_THFOOD50_LICENCE = True in the settings cell.
sys.path.insert(0, f"{CODE}/ml")
from gymfood.download import SOURCES
src = SOURCES["thfood50"]
print(src.title, "\n\n", src.licence, "\n\nTerms:", src.licence_url)
if not ACCEPT_THFOOD50_LICENCE:
    raise SystemExit("Stopped: set ACCEPT_THFOOD50_LICENCE = True after reading the terms, then Run all.")

In [ ]:
DIRS = ["--data-dir", f"{WORK}/data", "--runs-dir", f"{WORK}/runs", "--models-dir", f"{WORK}/models"]
S = f"{CODE}/ml/scripts"
# Backbone (Apache 2.0) and THFOOD-50 (~3.1 GB, checksummed). Skipped when already there.
run(sys.executable, f"{S}/download.py", "efficientnet_lite0", *DIRS)
run(sys.executable, f"{S}/download.py", "thfood50", "--accept-licence", "thfood50", *DIRS)

In [ ]:
# Map THFOOD-50 classes to our labels (classMap.json) and split. --no-personal: public data only.
run(sys.executable, f"{S}/prepare.py", "--no-personal", *DIRS)

In [ ]:
# Embeddings are cached in chunks and the head checkpoints every epoch: re-running resumes.
run(sys.executable, f"{S}/train.py", "--run-id", RUN_ID, "--epochs", str(EPOCHS), *DIRS)

In [ ]:
run(sys.executable, f"{S}/evaluate.py", "--run-id", RUN_ID, *DIRS)
print(open(f"{WORK}/runs/{RUN_ID}/report.md").read())

In [ ]:
# Export checks the model gives the trained head's answer on 200 images (export.min_top1_agreement).
# If export fails: read the "last output" above. The .tflite and its .json are still written, the
# trained head is kept, and re-running this cell only re-exports. Never lower the threshold in
# the config: a low agreement means the model in the app won't match the report.
MODEL = f"{WORK}/models/food-classifier-{RUN_ID}.tflite"
run(sys.executable, f"{S}/export.py", "--run-id", RUN_ID, "--out", MODEL, *DIRS)
print(open(MODEL.replace(".tflite", ".json")).read())

In [ ]:
# Download the model, then in the app: Settings → Food photos → Import model.
from google.colab import files
files.download(MODEL)